# Phase 7 - Business Interpretation and Retention

## Objective

Turn the validated final churn model into a transparent business-analysis layer focused on customer risk, churn drivers, and retention strategy review.

## 2. Load final model and cleaned data

This notebook uses the saved Phase 6 artifact without retraining the model or changing the selected threshold.

In [1]:
import os
import sys
import joblib
import numpy as np
import pandas as pd

project_root = r'c:\Users\aasye\Desktop\churn-prediction\churn-prediction'
if project_root not in sys.path:
    sys.path.insert(0, project_root)

artifact = joblib.load(os.path.join(project_root, 'models', 'final_churn_model.joblib'))
clean_df = pd.read_csv(os.path.join(project_root, 'data', 'processed', 'telco_churn_cleaned.csv'))
raw_df = pd.read_csv(os.path.join(project_root, 'data', 'raw', 'Telco-Customer-Churn.csv'))
print('model_name:', artifact.model_name)
print('threshold:', artifact.threshold)
print('rows:', len(clean_df))

model_name: Gradient Boosting
threshold: 0.35
rows: 7043


## 3. Generate model-based customer risk predictions

These outputs are for business interpretation and are not a new optimization pass.

In [2]:
analysis = clean_df.copy()
analysis['customerID'] = raw_df['customerID'].values
analysis['churn_probability'] = artifact.predict_proba(analysis.drop(columns=['Churn']))
analysis['predicted_churn'] = (analysis['churn_probability'] >= artifact.threshold).astype(int)
analysis['risk_tier'] = np.where(analysis['churn_probability'] < artifact.threshold, 'LOW', 'HIGH')
analysis[['customerID', 'churn_probability', 'predicted_churn', 'risk_tier']].head()

,customerID,churn_probability,predicted_churn,risk_tier
0,7590-VHVEG,0.647690,1,HIGH
1,5575-GNVDE,0.050520,0,LOW
2,3668-QPYBK,0.325404,0,LOW
3,7795-CFOCW,0.046661,0,LOW
4,9237-HQITU,0.669766,1,HIGH


## 4. Create risk tiers

LOW: probability < 0.35. HIGH: probability >= 0.35. The risk tier is a model-derived prioritization mechanism, not a guarantee that a customer will churn.

In [3]:
risk_counts = analysis['risk_tier'].value_counts().sort_index()
risk_counts

risk_tier
HIGH    2278
LOW     4765
Name: count, dtype: int64

## 5. Analyze high-risk customers

Compare customer/service characteristics between the predicted high-risk group and the lower-risk group without claiming causality.

In [4]:
high_risk = analysis[analysis['risk_tier'] == 'HIGH']
high_risk[['Contract', 'tenure', 'MonthlyCharges', 'InternetService', 'PaymentMethod', 'TechSupport', 'OnlineSecurity']].head()

,Contract,tenure,MonthlyCharges,InternetService,PaymentMethod,TechSupport,OnlineSecurity
0,Month-to-month,1,29.85,DSL,Electronic check,No,No
4,Month-to-month,2,70.70,Fiber optic,Electronic check,No,No
5,Month-to-month,8,99.65,Fiber optic,Electronic check,No,No
6,Month-to-month,22,89.10,Fiber optic,Credit card (automatic),No,No
8,Month-to-month,28,104.80,Fiber optic,Electronic check,Yes,No


## 6. Explain important churn drivers

The model uses a preprocessing pipeline and Gradient Boosting feature_importances_. This ranking identifies important model features but does not establish causal influence.

In [5]:
preprocessor = artifact.pipeline.named_steps['preprocessor']
model = artifact.pipeline.named_steps['model']
encoded_names = preprocessor.get_feature_names_out()
importance_df = pd.DataFrame({
    'feature': [name.split('__', 1)[1] if '__' in name else name for name in encoded_names],
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False).head(15)
importance_df

,feature,importance
37,Contract_Month-to-month,0.358276
0,tenure,0.178346
17,InternetService_Fiber optic,0.128110
19,OnlineSecurity_No,0.080518
28,TechSupport_No,0.059507
44,PaymentMethod_Electronic check,0.055361
1,MonthlyCharges,0.039045
2,TotalCharges,0.038932
13,MultipleLines_No,0.010455
39,Contract_Two year,0.009928


## 7. Compare high-risk vs lower-risk customer characteristics

Observed differences are interpreted as associations in a historical dataset; they are not proof of causation.

In [6]:
comparison = pd.DataFrame({
    'risk_tier': ['LOW', 'HIGH'],
    'customer_count': [len(analysis[analysis['risk_tier'] == 'LOW']), len(analysis[analysis['risk_tier'] == 'HIGH'])],
    'average_probability': [analysis.loc[analysis['risk_tier'] == 'LOW', 'churn_probability'].mean(), analysis.loc[analysis['risk_tier'] == 'HIGH', 'churn_probability'].mean()],
    'observed_churn_rate': [analysis.loc[analysis['risk_tier'] == 'LOW', 'Churn'].mean(), analysis.loc[analysis['risk_tier'] == 'HIGH', 'Churn'].mean()]
})
comparison

,risk_tier,customer_count,average_probability,observed_churn_rate
0,LOW,4765,0.117151,0.110178
1,HIGH,2278,0.575899,0.589991


## 8. Define retention decision logic

The goal is not to prescribe proven interventions. Instead, the logic maps risk and observable characteristics into a transparent retention review framework.

In [7]:
from src.retention_strategy import generate_retention_recommendation
sample_customer = {
    'Contract': 'Month-to-month',
    'tenure': 7,
    'MonthlyCharges': 82.0,
    'InternetService': 'Fiber optic',
    'OnlineSecurity': 'No',
    'TechSupport': 'No',
    'PaymentMethod': 'Electronic check',
    'PaperlessBilling': 'Yes'
}
generate_retention_recommendation(sample_customer, 0.74)

{'risk_tier': 'HIGH',
 'priority': 'high',
 'reason': 'Month-to-month contract indicates elevated churn risk.; Short tenure suggests early-life churn exposure.; High monthly charges are associated with elevated churn risk.; Fiber optic internet service is associated with higher churn risk.; Online security is not active, which is associated with higher churn risk.; Tech support is not active, which is associated with higher churn risk.; Electronic check payment is associated with higher churn risk.; Paperless billing is associated with elevated churn risk in this historical sample.',
 'recommended_action': 'Potential contract-retention outreach and plan review.; Prioritize onboarding and value-check outreach.; Review pricing, included value, and plan-fit options.; Review connectivity value and service satisfaction touchpoints.; Discuss security-product adoption and digital protection offer.; Offer targeted support review and service guidance.; Discuss automatic payment enrollment and p

## 9. Generate retention recommendations

The recommendation function uses deterministic rules based only on available customer attributes and model probability.

In [8]:
analysis['priority'] = analysis.apply(lambda row: generate_retention_recommendation(row.to_dict(), row['churn_probability'])['priority'], axis=1)
analysis['reason'] = analysis.apply(lambda row: generate_retention_recommendation(row.to_dict(), row['churn_probability'])['reason'], axis=1)
analysis['recommended_action'] = analysis.apply(lambda row: generate_retention_recommendation(row.to_dict(), row['churn_probability'])['recommended_action'], axis=1)
analysis[['customerID', 'risk_tier', 'priority', 'reason', 'recommended_action']].head()

,customerID,risk_tier,priority,reason,recommended_action
0,7590-VHVEG,HIGH,high,Month-to-month contract indicates elevated chu...,Potential contract-retention outreach and plan...
1,5575-GNVDE,LOW,low,Longer-term contract is associated with lower ...,Monitor account and maintain routine service r...
2,3668-QPYBK,LOW,low,Month-to-month contract indicates elevated chu...,Monitor account and maintain routine service r...
3,7795-CFOCW,LOW,low,Longer-term contract is associated with lower ...,Monitor account and maintain routine service r...
4,9237-HQITU,HIGH,high,Month-to-month contract indicates elevated chu...,Potential contract-retention outreach and plan...


## 10. Analyze limitations and responsible interpretation

This is a historical Telco dataset, the model is probabilistic, feature importance is not causal, and the recommendations are rule-based review suggestions rather than proven retention interventions.

## 11. Export business-analysis results

The customer-level output and summary are saved for the later prediction interface and business review.

In [9]:
risk_columns = ['customerID', 'churn_probability', 'predicted_churn', 'risk_tier', 'priority', 'reason', 'recommended_action']
analysis[risk_columns].sort_values('churn_probability', ascending=False).to_csv(os.path.join(project_root, 'reports', 'phase7_customer_risk_analysis.csv'), index=False)
summary = analysis.groupby('risk_tier').agg(
    customer_count=('customerID', 'count'),
    average_churn_probability=('churn_probability', 'mean'),
    predicted_churn_rate=('predicted_churn', 'mean'),
    observed_churn_rate=('Churn', 'mean'),
).reset_index()
summary['percentage_of_customers'] = summary['customer_count'] / len(analysis)
summary.to_csv(os.path.join(project_root, 'reports', 'phase7_retention_summary.csv'), index=False)
summary

,risk_tier,customer_count,average_churn_probability,predicted_churn_rate,observed_churn_rate,percentage_of_customers
0,HIGH,2278,0.575899,1.0,0.589991,0.323442
1,LOW,4765,0.117151,0.0,0.110178,0.676558


## 12. Summary

The final model remains Gradient Boosting at threshold 0.35. Phase 7 adds a transparent business-analysis layer without changing the model, threshold, or data pipeline.